In [ ]:
!pip install pymanopt

In [ ]:
%reset -f

In [ ]:
import numpy as np
# import tensorflow as tf
import cupy as cp
# import jax.numpy as jnp
# import jax
# from jax import jit, lax
# from jax import device_put
# from jax.lib import xla_bridge

# import pymanopt
# from pymanopt.manifolds import Oblique
# from pymanopt.optimizers import TrustRegions
#from cupy.dtypes import float8_e4m3fn, float8_e5m2

import time
import gc

In [ ]:
import os

# 2. Use CUB and cuTENSOR backends (Faster reductions/elementwise)
os.environ["CUPY_ACCELERATORS"] = "cub,cutensor"

# 3. Tune the Compiler
# Tells the GPU compiler to optimize for the A100 architecture specifically
os.environ["CUDA_CACHE_MAXSIZE"] = "2147483648" # 2GB Cache

# # TF32 settings
# # Allow CuPy/CUDA to use Tensor Cores for Float32
# os.environ["NVIDIA_TF32_OVERRIDE"] = "1"

In [ ]:
def reset_memory():
    # Delete all possible GPU arrays
    globals_ = list(globals().keys())
    for var in globals_:
        if var in ['cp', 'jnp', 'reset_memory']: continue
        if isinstance(globals()[var], (cp.ndarray, jnp.ndarray)):
            del globals()[var]

    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

In [ ]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))
shift = 50/n

C = cp.random.randn(n,n)/n
C = C + C.T
cp.fill_diagonal(C, C.diagonal() + shift)

# generate_B.seed = 0
# B = generate_B.randn(k,n)
B = cp.random.randn(k,n)
B = B/cp.linalg.norm(B,axis = 0)

# cp_f8 = float8_e5m2
# cp_f8 = float8_e4m3fn

C_32 = cp.array(C,dtype = cp.float32)
B_32 = cp.array(B,dtype = cp.float32)

B_32.dot(C_32)
cp.cuda.Device().synchronize()
# 1000 is good enough do not exceed 1500
K32 = [100,100,100,100,100,100,100,100,100,100]
for i in range(len(K32)):
  cg0 = time.time()
  for t in range(K32[i]):
      B_32 = B_32.dot(C_32)
      B_32 = B_32/cp.linalg.norm(B_32,axis = 0)
  cp.cuda.Device().synchronize()
  cg1 = time.time()
  obj64 = cp.tensordot(B_32,B_32.dot(C))
  print("block time",cg1-cg0)
  print("obj 64",obj64)

block time 2.591982126235962
obj 64 531.5577717260352
block time 2.5614755153656006
obj 64 532.1124742864254
block time 2.5662012100219727
obj 64 532.1871159271236
block time 2.5658297538757324
obj 64 532.2071538534698
block time 2.5700395107269287
obj 64 532.2147071223914
block time 2.568537950515747
obj 64 532.2181971520478
block time 2.5728273391723633
obj 64 532.2200436237022
block time 2.5745482444763184
obj 64 532.2211183740674
block time 2.5754551887512207
obj 64 532.2217895784612
block time 2.5763351917266846
obj 64 532.2222307563848


In [ ]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))

repeat = 50

f_cg = np.zeros((repeat,10))
t_cg = np.zeros((repeat,10))

shift = 50/n

cp.random.seed(0)
for rep in range(repeat):
  print("----------------REP++",rep,"++---------------------------------------")
  # reset_memory()
  C = cp.random.randn(n,n)/n
  C = C + C.T
  cp.fill_diagonal(C, C.diagonal() + shift)

  # generate_B.seed = 0
  # B = generate_B.randn(k,n)
  B = cp.random.randn(k,n)
  B = B/cp.linalg.norm(B,axis = 0)

  C_32 = cp.array(C,dtype = cp.float32)
  B_32 = cp.array(B,dtype = cp.float32)
  dummy_res = B_32.dot(C_32)
  cp.cuda.Device().synchronize()

  # 1000 is good enough do not exceed 1500
  K32 = [100,100,100,100,100,100,100,100,100,100]
  for i in range(len(K32)):
    cg0 = time.time()
    for t in range(K32[i]):
        B_32 = B_32.dot(C_32)
        B_32 = B_32/cp.linalg.norm(B_32,axis = 0)
    cp.cuda.Device().synchronize()
    cg1 = time.time()

    # obj64 = cp.tensordot(B_32,B_32.dot(C))-shift *n
    obj64 = cp.tensordot(B_32,B_32.dot(C))
    print("block time",cg1-cg0)
    print("obj 64",obj64)
    f_cg[rep,i] = obj64
    t_cg[rep,i] = cg1-cg0

----------------REP++ 0 ++---------------------------------------
block time 2.5737595558166504
obj 64 531.7209759424513
block time 2.578335762023926
obj 64 532.2614603337167
block time 2.5783956050872803
obj 64 532.33383270591
block time 2.580544948577881
obj 64 532.3532177749305
block time 2.5790388584136963
obj 64 532.3605266273772
block time 2.5804967880249023
obj 64 532.3638978498628
block time 2.5806877613067627
obj 64 532.3656737499502
block time 2.5815412998199463
obj 64 532.3667024327782
block time 2.5810327529907227
obj 64 532.3673422347224
block time 2.5825488567352295
obj 64 532.3677626732218
----------------REP++ 1 ++---------------------------------------
block time 2.58203387260437
obj 64 531.7590150974877
block time 2.5833358764648438
obj 64 532.3146765703739
block time 2.582226514816284
obj 64 532.3884625222119
block time 2.582390069961548
obj 64 532.407996602441
block time 2.5835461616516113
obj 64 532.4153136583124
block time 2.5830495357513428
obj 64 532.41869036344

In [ ]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu_float32_final.npz",f_cg = f_cg, t_cg = t_cg)

In [ ]:
np.cumsum(t_cg.mean(axis = 0))

array([ 2.58679798,  5.17643113,  7.76580514, 10.35517954, 12.94455862,
       15.53419317, 18.12362259, 20.7132706 , 23.30292863, 25.89280045])

In [ ]:
f_cg.mean(axis = 0)

array([531.6867906 , 532.23218515, 532.30480641, 532.32425494,
       532.33161723, 532.33503397, 532.33684515, 532.33789927,
       532.33855654, 532.33898887])

In [ ]:
print(cp.__version__)

14.0.1


In [ ]:
!nvidia-smi

Mon Mar  2 19:20:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   61C    P0             84W /  400W |   34780MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----